In [1]:
import numpy as np
import pandas as pd
import json
from datetime import datetime
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

In [2]:
# -------------------- DATA LOADING --------------------
review_df = pd.read_parquet("../data/output/review_data.parquet")
meta_df = pd.read_parquet("../data/output/meta_data.parquet")

In [3]:
meta_df.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Appstore for Android,Accupressure Guide,3.6,NaN,[All the pressing point has been explained wit...,[Acupressure technique is very ancient and ver...,0.00,"[{'hi_res': None, 'large': 'https://m.media-am...","[{'title': '', 'url': '', 'user_id': ''}]",mAppsguru,[],"{'': None, 'Age Range (Description)': None, 'A...",B00VRPSGEO,None,NaN,NaN
1,Appstore for Android,Ankylosaurus Fights Back - Smithsonian's Prehi...,4.0,NaN,[ENCOURAGE literacy skills with highlighted na...,[Join Ankylosaurus in this interactive book ap...,2.99,"[{'hi_res': None, 'large': 'https://m.media-am...","[{'title': '', 'url': '', 'user_id': ''}]","Oceanhouse Media, Inc",[],"{'': None, 'Age Range (Description)': None, 'A...",B00NWQXXHQ,None,NaN,NaN
2,Appstore for Android,Mahjong 2015,3.1,NaN,[Mahjong 2015 is a free solitaire matching gam...,[Mahjong 2015 is a free solitaire matching gam...,0.00,"[{'hi_res': None, 'large': 'https://m.media-am...","[{'title': '', 'url': '', 'user_id': ''}]",sophiathach,[],"{'': None, 'Age Range (Description)': None, 'A...",B00RFKP6AC,None,NaN,NaN
3,Appstore for Android,Jewels Brick Breakout,4.2,NaN,"[Game Features:, - Intuitive touch controls wi...",[Jewels Brick Breakout is a glowing jewels bri...,0.00,"[{'hi_res': None, 'large': 'https://m.media-am...","[{'title': '', 'url': '', 'user_id': ''}]",Bad Chicken,[],"{'': None, 'Age Range (Description)': None, 'A...",B00SP2QU0E,None,NaN,NaN
4,Appstore for Android,Traffic Police: Off-Road Cub,3.3,NaN,"[In this game you will find:, - Killer police ...",[Become the best road police officer in Cube C...,0.00,"[{'hi_res': None, 'large': 'https://m.media-am...","[{'title': '', 'url': '', 'user_id': ''}]",Dast 2 For Metro,[],"{'': None, 'Age Range (Description)': None, 'A...",B01DZIT64O,None,NaN,NaN


In [3]:
# Add review_id as column (not index)
review_df = review_df.reset_index(names="review_id")

In [4]:
# -------------------- PREPROCESSING --------------------
# Convert timestamp to datetime (critical for temporal features)
review_df['review_date'] = pd.to_datetime(review_df['timestamp'], unit='ms')

# Extract text features
review_df['review_text_length'] = review_df['text'].str.len()
review_df['review_title_length'] = review_df['title'].str.len()
review_df['review_word_count'] = review_df['text'].str.split().str.len()

# Sentiment proxy (simple heuristic)
review_df['is_extreme_rating'] = review_df['rating'].isin([1.0, 5.0])
review_df['is_positive'] = review_df['rating'] >= 4.0
review_df['is_negative'] = review_df['rating'] <= 2.0

# Image counts
review_df['num_review_images'] = review_df['images'].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)

In [5]:
review_df.head(5)

,review_id,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,review_date,review_text_length,review_title_length,review_word_count,is_extreme_rating,is_positive,is_negative,num_review_images
0,0,1.0,malware,mcaffee IS malware,[],B07BFS3G7P,B0BQSK9QCF,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1562182632076,0,False,2019-07-03 19:37:12.076,18,7,3,True,False,True,0
1,1,5.0,Lots of Fun,I love playing tapped out because it is fun to...,[],B00CTQ6SIG,B00CTQ6SIG,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,1424120336000,0,True,2015-02-16 20:58:56.000,140,11,26,True,True,False,0
2,2,5.0,Light Up The Dark,I love this flashlight app! It really illumin...,[],B0066WJLU6,B0066WJLU6,AHSPLDNW5OOUK2PLH7GXLACFBZNQ,1362399267000,0,True,2013-03-04 12:14:27.000,112,17,20,True,True,False,0
3,3,4.0,Fun game,One of my favorite games,[],B00KCYMAWK,B00KCYMAWK,AH6CATODIVPVUOJEWHRSRCSKAOHA,1561061428662,0,True,2019-06-20 20:10:28.662,24,8,5,False,True,False,0
4,4,4.0,I am not that good at it but my kids are,Cute game. I am not that good at it but my kid...,[],B00P1RK566,B00P1RK566,AEINY4XOINMMJCK5GZ3M6MMHBN6A,1418257196000,0,True,2014-12-11 00:19:56.000,74,40,17,False,True,False,0


In [6]:
# -------------------- MERGE WITH METADATA --------------------
# not dropping NaN prices
merged_df = review_df.merge(meta_df, on='parent_asin', how='left')

# Handle price properly
merged_df['price'] = merged_df['price'].fillna(-1)  # -1 indicates unknown
merged_df['is_free'] = merged_df['price'] == 0.0
merged_df['has_price_info'] = merged_df['price'] >= 0
merged_df['price_bucket'] = pd.cut(
    merged_df['price'],
    bins=[-1, 0, 10, 25, 50, 100, 1000, 2000],
    labels=['unknown', 'free', '0-10', '10-25', '25-50', '50-100', '100+']
)

# Extract metadata features
def safe_len(x):
    """Safely get length of arrays"""
    if isinstance(x, (list, np.ndarray)):
        return len(x)
    return 0

merged_df['num_features'] = merged_df['features'].apply(safe_len)
merged_df['num_descriptions'] = merged_df['description'].apply(safe_len)
merged_df['num_categories'] = merged_df['categories'].apply(safe_len)

# Extract primary category (first in list)
merged_df['primary_category'] = merged_df['categories'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'unknown'
)

In [7]:
# -------------------- TEMPORAL FEATURES --------------------
# Calculate days since review (from most recent date in dataset)
max_date = merged_df['review_date'].max()
merged_df['days_since_review'] = (max_date - merged_df['review_date']).dt.days
merged_df['review_year'] = merged_df['review_date'].dt.year
merged_df['review_month'] = merged_df['review_date'].dt.month

# Recency weight (exponential decay with 1-year half-life)
merged_df['recency_weight'] = np.exp(-merged_df['days_since_review'] / 365.25)

In [ ]:
# -------------------- TRAIN/VAL/TEST SPLIT --------------------
# Temporal split 
train_df = merged_df[merged_df['review_date'] < '2022-01-01']
val_df = merged_df[
    (merged_df['review_date'] >= '2022-01-01') &
    (merged_df['review_date'] < '2023-01-01')
]
test_df = merged_df[merged_df['review_date'] >= '2023-01-01']

print(f"Train: {len(train_df)} reviews ({train_df['review_date'].min()} to {train_df['review_date'].max()})")
print(f"Val: {len(val_df)} reviews")
print(f"Test: {len(test_df)} reviews")

Train: 4698506 reviews (1999-03-15 04:02:39 to 2021-12-31 23:58:00.244000)
Val: 148905 reviews
Test: 32770 reviews


In [9]:
# -------------------- USER FEATURES --------------------
# Build comprehensive user profile (ONE ROW PER USER)

user_features = (
    train_df[train_df['verified_purchase'] == True]  # Only verified for training
    .groupby('user_id')
    .agg({
        # Basic stats
        'review_id': 'count',
        'rating': ['mean', 'std', 'min', 'max'],

        # Temporal
        'review_date': ['min', 'max'],
        'recency_weight': 'sum',

        # Quality signals
        'helpful_vote': ['sum', 'mean'],
        'verified_purchase': 'sum',

        # Text engagement
        'review_text_length': 'mean',
        'review_word_count': 'mean',
        'num_review_images': 'sum',

        # Rating patterns
        'is_extreme_rating': 'mean',
        'is_positive': 'mean',
        'is_negative': 'mean',

        # Price sensitivity
        'price': 'mean',
        'is_free': 'mean'
    })
)

# Flatten multi-level columns
user_features.columns = ['_'.join(col).strip('_') for col in user_features.columns]
user_features = user_features.reset_index()

# Rename for clarity
user_features = user_features.rename(columns={
    'review_id_count': 'num_reviews',
    'rating_mean': 'avg_rating_given',
    'rating_std': 'rating_std',
    'rating_min': 'min_rating_given',
    'rating_max': 'max_rating_given',
    'review_date_min': 'first_review_date',
    'review_date_max': 'last_review_date',
    'recency_weight_sum': 'total_recency_weight',
    'helpful_vote_sum': 'total_helpful_votes_received',
    'helpful_vote_mean': 'avg_helpful_votes_per_review',
    'verified_purchase_sum': 'num_verified_purchases',
    'review_text_length_mean': 'avg_review_length',
    'review_word_count_mean': 'avg_review_words',
    'num_review_images_sum': 'total_review_images',
    'is_extreme_rating_mean': 'extreme_rating_ratio',
    'is_positive_mean': 'positive_rating_ratio',
    'is_negative_mean': 'negative_rating_ratio',
    'price_mean': 'avg_price_purchased',
    'is_free_mean': 'free_app_ratio'
})

# Derive additional features
user_features['days_active'] = (
    user_features['last_review_date'] - user_features['first_review_date']
).dt.days + 1

user_features['reviews_per_day'] = (
    user_features['num_reviews'] / user_features['days_active']
)

user_features['verified_purchase_ratio'] = (
    user_features['num_verified_purchases'] / user_features['num_reviews']
)

# User segmentation (from EDA)
user_features['user_segment'] = pd.cut(
    user_features['num_reviews'],
    bins=[0, 1, 10, np.inf],
    labels=['one_time', 'occasional', 'power_user']
)

# Rating discriminativeness (higher std = more discriminating)
user_features['is_discriminating'] = user_features['rating_std'] > 1.0

print(f"Total users: {len(user_features)}")
print(f"User segments:\n{user_features['user_segment'].value_counts()}")

Total users: 2359517
User segments:
user_segment
one_time      1578086
occasional     752310
power_user      29121
Name: count, dtype: int64


In [10]:
# -------------------- ITEM FEATURES --------------------
# Build comprehensive item profile (ONE ROW PER ITEM)

item_features = (
    train_df
    .groupby('parent_asin')
    .agg({
        # Basic stats
        'user_id': 'count',
        'rating': ['mean', 'std', 'min', 'max'],

        # Temporal
        'review_date': ['min', 'max'],
        'recency_weight': 'sum',

        # Quality signals
        'helpful_vote': ['sum', 'mean', 'max'],
        'verified_purchase': 'sum',

        # Engagement
        'review_text_length': 'mean',
        'review_word_count': 'mean',

        # Rating patterns
        'is_positive': 'mean',
        'is_negative': 'mean',

        # Metadata (take first/mode)
        'main_category': 'first',
        'primary_category': 'first',
        'store': 'first',
        'price': 'first',
        'is_free': 'first',
        'num_features': 'first',
        'num_descriptions': 'first',
        'num_categories': 'first'
    })
)

# Flatten columns
item_features.columns = ['_'.join(col).strip('_') for col in item_features.columns]
item_features = item_features.reset_index()

# Rename
item_features = item_features.rename(columns={
    'user_id_count': 'num_reviews',
    'rating_mean': 'avg_rating',
    'rating_std': 'rating_std',
    'rating_min': 'min_rating',
    'rating_max': 'max_rating',
    'review_date_min': 'first_review_date',
    'review_date_max': 'last_review_date',
    'recency_weight_sum': 'total_recency_weight',
    'helpful_vote_sum': 'total_helpful_votes',
    'helpful_vote_mean': 'avg_helpful_votes',
    'helpful_vote_max': 'max_helpful_votes',
    'verified_purchase_sum': 'num_verified_reviews',
    'review_text_length_mean': 'avg_review_length_received',
    'review_word_count_mean': 'avg_review_words_received',
    'is_positive_mean': 'positive_review_ratio',
    'is_negative_mean': 'negative_review_ratio'
})

# Clean up column names (remove '_first')
for col in item_features.columns:
    if col.endswith('_first'):
        item_features = item_features.rename(columns={col: col.replace('_first', '')})

# Derive features
item_features['days_on_platform'] = (
    item_features['last_review_date'] - item_features['first_review_date']
).dt.days + 1

item_features['reviews_per_day'] = (
    item_features['num_reviews'] / item_features['days_on_platform']
)

item_features['verified_review_ratio'] = (
    item_features['num_verified_reviews'] / item_features['num_reviews']
)

# Item popularity segment (from EDA)
item_features['popularity_segment'] = pd.cut(
    item_features['num_reviews'],
    bins=[0, 1, 10, 100, np.inf],
    labels=['cold_start', 'low_coverage', 'medium', 'popular']
)

# Quality score (Wilson lower bound for rating confidence)
def wilson_lower_bound(pos, n, confidence=0.95):
    """Wilson score interval for rating confidence"""
    if n == 0:
        return 0
    z = 1.96  # 95% confidence
    phat = pos / n
    return (phat + z*z/(2*n) - z * np.sqrt((phat*(1-phat)+z*z/(4*n))/n))/(1+z*z/n)

item_features['quality_score'] = item_features.apply(
    lambda row: wilson_lower_bound(
        row['positive_review_ratio'] * row['num_reviews'],
        row['num_reviews']
    ),
    axis=1
)

print(f"Total items: {len(item_features)}")
print(f"Popularity segments:\n{item_features['popularity_segment'].value_counts()}")

Total items: 86239
Popularity segments:
popularity_segment
low_coverage    41510
cold_start      24141
medium          15855
popular          4733
Name: count, dtype: int64


In [11]:
# -------------------- USER-ITEM MATRIX --------------------
# Create sparse matrices for collaborative filtering

def create_interaction_matrix(df, value_col='rating'):
    """Create sparse user-item interaction matrix"""
    # Create mappings
    user_ids = df['user_id'].unique()
    item_ids = df['parent_asin'].unique()

    user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
    item_to_idx = {iid: idx for idx, iid in enumerate(item_ids)}

    # Map to indices
    user_indices = df['user_id'].map(user_to_idx)
    item_indices = df['parent_asin'].map(item_to_idx)
    values = df[value_col].values

    # Create sparse matrix
    matrix = csr_matrix(
        (values, (user_indices, item_indices)),
        shape=(len(user_ids), len(item_ids))
    )

    return matrix, user_to_idx, item_to_idx

# Create explicit rating matrix
train_matrix, user_to_idx, item_to_idx = create_interaction_matrix(train_df)
print(f"Train matrix shape: {train_matrix.shape}")
print(f"Sparsity: {100 * (1 - train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1])):.4f}%")

# Create binary implicit feedback matrix (1 if interaction occurred)
train_df['implicit_feedback'] = 1
train_implicit, _, _ = create_interaction_matrix(train_df, value_col='implicit_feedback')

# Create weighted matrix (rating * recency_weight for temporal decay)
train_df['weighted_rating'] = train_df['rating'] * train_df['recency_weight']
train_weighted, _, _ = create_interaction_matrix(train_df, value_col='weighted_rating')

print(f"\n✅ Created 3 interaction matrices:")
print(f"  - train_matrix: Explicit ratings (1.0-5.0)")
print(f"  - train_implicit: Binary feedback (0 or 1)")
print(f"  - train_weighted: Time-decayed ratings")

Train matrix shape: (2498956, 86239)
Sparsity: 99.9978%


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25012\1060083195.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['implicit_feedback'] = 1
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_25012\1060083195.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['weighted_rating'] = train_df['rating'] * train_df['recency_weight']



✅ Created 3 interaction matrices:
  - train_matrix: Explicit ratings (1.0-5.0)
  - train_implicit: Binary feedback (0 or 1)
  - train_weighted: Time-decayed ratings


In [12]:
# -------------------- CATEGORY FEATURES --------------------
# Handle categories separately (many-to-many relationship)

# User-Category preferences
user_category_features = (
    train_df
    .explode('categories')  # NOW we explode (after main aggregations)
    .groupby(['user_id', 'categories'])
    .agg({
        'review_id': 'count',
        'rating': 'mean'
    })
    .reset_index()
    .rename(columns={
        'categories': 'category',
        'review_id': 'num_reviews_in_category',
        'rating': 'avg_rating_in_category'
    })
)

# Calculate user's favorite categories
user_top_categories = (
    user_category_features
    .sort_values(['user_id', 'num_reviews_in_category'], ascending=[True, False])
    .groupby('user_id')
    .head(3)  # Top 3 categories per user
)

# Item-Category mapping (for content-based filtering)
item_category_features = (
    meta_df[['parent_asin', 'categories', 'main_category']]
    .explode('categories')
    .rename(columns={'categories': 'category'})
)

print(f"User-category pairs: {len(user_category_features)}")
print(f"Item-category pairs: {len(item_category_features)}")
print(f"Unique categories: {user_category_features['category'].nunique()}")

User-category pairs: 1124939
Item-category pairs: 126239
Unique categories: 312


In [13]:
# -------------------- SAVE EVERYTHING --------------------
import os
import pickle
from scipy.sparse import save_npz

output_dir = "../data/processed/"
os.makedirs(output_dir, exist_ok=True)

# Helper function to save both parquet and CSV
def save_dataframe(df, name, output_dir=output_dir):
    """Save dataframe in both parquet and CSV formats"""
    parquet_path = f"{output_dir}{name}.parquet"
    csv_path = f"{output_dir}{name}.csv"

    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)

    print(f"  Saved {name}: {len(df):,} rows")
    return parquet_path, csv_path

# -------------------- 1. ENRICHED SOURCE DATA --------------------
print("Saving enriched source data...")

# Enriched review dataframe (with text features, sentiment proxies, etc.)
save_dataframe(review_df, "review_data_enriched")

# Enriched merged dataframe (reviews + metadata with all derived features)
save_dataframe(merged_df, "merged_data_enriched")

# -------------------- 2. FEATURE DATAFRAMES --------------------
print("\nSaving feature dataframes...")

save_dataframe(user_features, "user_features")
save_dataframe(item_features, "item_features")
save_dataframe(user_category_features, "user_category_features")
save_dataframe(item_category_features, "item_category_features")

# -------------------- 3. TRAIN/VAL/TEST SPLITS --------------------
print("\nSaving train/val/test splits...")

save_dataframe(train_df, "train_interactions")
save_dataframe(val_df, "val_interactions")
save_dataframe(test_df, "test_interactions")

# -------------------- 4. ID MAPPINGS --------------------
print("\nSaving ID mappings...")

# Save as pickle (for Python)
with open(f"{output_dir}user_to_idx.pkl", 'wb') as f:
    pickle.dump(user_to_idx, f)
with open(f"{output_dir}item_to_idx.pkl", 'wb') as f:
    pickle.dump(item_to_idx, f)

# Also save as CSV for portability
user_mapping_df = pd.DataFrame([
    {'user_id': uid, 'user_idx': idx}
    for uid, idx in user_to_idx.items()
])
item_mapping_df = pd.DataFrame([
    {'parent_asin': iid, 'item_idx': idx}
    for iid, idx in item_to_idx.items()
])
user_mapping_df.to_csv(f"{output_dir}user_to_idx.csv", index=False)
item_mapping_df.to_csv(f"{output_dir}item_to_idx.csv", index=False)

print(f"  Saved user_to_idx: {len(user_to_idx):,} users (.pkl + .csv)")
print(f"  Saved item_to_idx: {len(item_to_idx):,} items (.pkl + .csv)")

# -------------------- 5. SPARSE MATRICES --------------------
print("\nSaving sparse matrices...")

save_npz(f"{output_dir}train_matrix.npz", train_matrix)
save_npz(f"{output_dir}train_implicit.npz", train_implicit)
save_npz(f"{output_dir}train_weighted.npz", train_weighted)

print(f"  Saved train_matrix.npz: {train_matrix.shape}")
print(f"  Saved train_implicit.npz: {train_implicit.shape}")
print(f"  Saved train_weighted.npz: {train_weighted.shape}")

# -------------------- 6. METADATA / SUMMARY --------------------
print("\nSaving processing metadata...")

# Save a summary of the processing for reproducibility
processing_summary = {
    'processing_date': datetime.now().isoformat(),
    'source_files': {
        'review_data': '../data/output/review_data.parquet',
        'meta_data': '../data/output/meta_data.parquet'
    },
    'dataset_stats': {
        'total_reviews': len(merged_df),
        'total_users': len(user_features),
        'total_items': len(item_features),
        'train_size': len(train_df),
        'val_size': len(val_df),
        'test_size': len(test_df),
        'train_date_range': f"< 2022-01-01",
        'val_date_range': f"2022-01-01 to 2022-12-31",
        'test_date_range': f">= 2023-01-01"
    },
    'feature_counts': {
        'user_features': len(user_features.columns),
        'item_features': len(item_features.columns)
    },
    'matrix_stats': {
        'shape': train_matrix.shape,
        'nnz': train_matrix.nnz,
        'sparsity_pct': 100 * (1 - train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1]))
    }
}

with open(f"{output_dir}processing_summary.json", 'w') as f:
    json.dump(processing_summary, f, indent=2, default=str)

print(f"  Saved processing_summary.json")

# -------------------- FINAL SUMMARY --------------------
print("\n" + "="*60)
print("✅ ALL PROCESSED DATA SAVED TO:", output_dir)
print("="*60)

print(f"""
📁 Files created:

ENRICHED DATA (parquet + csv):
  - review_data_enriched     ({len(review_df):,} reviews)
  - merged_data_enriched     ({len(merged_df):,} reviews with metadata)

FEATURE TABLES (parquet + csv):
  - user_features            ({len(user_features):,} users, {len(user_features.columns)} features)
  - item_features            ({len(item_features):,} items, {len(item_features.columns)} features)
  - user_category_features   ({len(user_category_features):,} user-category pairs)
  - item_category_features   ({len(item_category_features):,} item-category pairs)

TRAIN/VAL/TEST SPLITS (parquet + csv):
  - train_interactions       ({len(train_df):,} reviews, <2022)
  - val_interactions         ({len(val_df):,} reviews, 2022)
  - test_interactions        ({len(test_df):,} reviews, 2023+)

ID MAPPINGS (pkl + csv):
  - user_to_idx              ({len(user_to_idx):,} users)
  - item_to_idx              ({len(item_to_idx):,} items)

SPARSE MATRICES (npz):
  - train_matrix             (explicit ratings)
  - train_implicit           (binary feedback)
  - train_weighted           (time-decayed ratings)

METADATA:
  - processing_summary.json  (reproducibility info)
""")

Saving enriched source data...
  Saved review_data_enriched: 4,880,181 rows


ArrowMemoryError: realloc of size 3758096384 failed